In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set_theme()
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.getcwd()
os.chdir('/content/drive/MyDrive/Colab Notebooks/M0572-SistemesAprenentatgeAutomatic/')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/M0572-SistemesAprenentatgeAutomatic/'

In [ ]:

path = "/content/drive/MyDrive/Colab Notebooks/M0572-SistemesAprenentatgeAutomatic/alumnat_share/titanic/Titanic-Dataset.csv"
df = pd.read_csv(path)
df.head()

In [ ]:
#Duplicats

print(df.duplicated().sum())


In [ ]:
#Nulls
df.isnull().sum()


In [ ]:
# Eliminem la columna Cabin, no ens dona informació important (només ens podria donar si són de classe alta i això ho obtenim de Fare, ticket o Pclass)
df = df.drop(columns=['Cabin'])

In [ ]:
from sklearn.impute import KNNImputer
# KNNImputer per tenir la mitjana d'edat dels veïns dins la taula i reomplir Age
imp = KNNImputer(n_neighbors=10)
df[['Age']] = imp.fit_transform(df[['Age']])


In [ ]:
# Posem la moda als 2 nulls d'Embarked
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [ ]:
#Nulls
df.isnull().sum()

In [ ]:
# Creació de noves columnes
df["Is_male"] = (df["Sex"] == "male").astype(int)
df = df.drop(columns=["Sex"])


In [ ]:
# Passem Embarked a dummies, LRM no acceptara el valor en format string
df = pd.get_dummies(df, columns=["Embarked"], drop_first=False,dtype=int)


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# Quants han sobreviscut? 0: mort 1: viu
df.Survived.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test = train_test_split(df, test_size = 0.20, random_state = 0)
X_train.shape, X_test.shape

In [ ]:
#predictors = df.columns[[2,4,5,6,8,9,10,11,12]]
predictors = df.columns[[2,4,9]]
predictors

In [ ]:
to_predict = df.columns[[1]]
to_predict

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000).fit(X_train[predictors], X_train[to_predict].to_numpy().ravel())

In [ ]:
X_test['predict'] = model.predict(X_test[predictors])

In [ ]:
from sklearn.metrics import f1_score
# Coeficients
print(f'\u03B21: {0}, \u03B22: {1}, ',model.coef_.item(0),model.coef_.item(1))
# Intercept (x = 0)
print('\u03B20: %.5f' % model.intercept_.item())
# error
print('R^2:', model.score(X_test[predictors], X_test[to_predict]))

y_true = X_test[to_predict]
y_pred = model.predict(X_test[predictors])

print("F1:", f1_score(y_true, y_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
sns.set_theme(context = 'notebook', style = "white", font_scale = 1.0)
_, axs = plt.subplots(1, 3, figsize = (14, 4), sharey = True)
titles = [
    "Sense normalitzar (valors absoluts)",
    "files (true) — mostra el RECALL per classe",
    "columnes (pred) — mostra la PRECISION per classe"
]
normalizations = [None, 'true', 'pred']
#for i, norm in enumerate([None, 'true', 'pred']):
for ax, norm, title in zip(axs.flatten(), normalizations, titles):
    ConfusionMatrixDisplay.from_predictions(
        X_test.Survived,
        X_test.predict,
        normalize = norm,
        values_format = '.2f' if norm is not None else '.0f',
        ax = ax,
        colorbar = False,
        cmap = 'GnBu'
    )
    ax.set_title(title)
plt.tight_layout();

Que tenim?
---


---

- 92 --> teniem 92 morts que els ha predit correctament

- 18 --> teniem 18 morts de més que els ha predit com a vius

- 18 --> teniem 18 vius que ha predit com a morts

- 51 --> teniem 51 vius que ha predit correctament

---

- El 84% dels morts els hem predit com a morts
- El 16% dels morts els hem classificat malament
- El 74% dels vius els hem predit bé
- El 26% dels vius els hem classificat com a morts


In [ ]:
X_test.predict.value_counts()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_scaled = pd.DataFrame(scaler.fit_transform(df[predictors]), columns = predictors)
df_scaled[to_predict] = df[to_predict]

df_scaled.describe()


In [ ]:
Z_train, Z_test = train_test_split(df_scaled, test_size = 0.2, random_state = 1234)
model_scaled = LogisticRegression().fit(Z_train[predictors], Z_train[to_predict].to_numpy().ravel())

In [ ]:
Z_test['predict'] = model_scaled.predict(Z_test[predictors])

In [ ]:
# Coeficients
print(f'\u03B21: {0}, \u03B22: {1}, ',model_scaled.coef_.item(0),model_scaled.coef_.item(1))
# Intercept (x = 0)
print('\u03B20: %.5f' % model_scaled.intercept_.item())
# error
print('R^2:', model_scaled.score(Z_test[predictors], Z_test[to_predict]))

In [ ]:
Z_test.predict.value_counts()

In [ ]:
_, axs = plt.subplots(1, 3, figsize = (14, 4), sharey = True)
titles = [
    "Sense normalitzar (valors absoluts)",
    "files (true) — mostra el RECALL per classe",
    "columnes (pred) — mostra la PRECISION per classe"
]
normalizations = [None, 'true', 'pred']
#for i, norm in enumerate([None, 'true', 'pred']):
for ax, norm, title in zip(axs.flatten(), normalizations, titles):
    ConfusionMatrixDisplay.from_predictions(
        X_test.Survived,
        X_test.predict,
        normalize = norm,
        values_format = '.2f' if norm is not None else '.0f',
        ax = ax,
        colorbar = False,
        cmap = 'GnBu'
    )
    ax.set_title(title)
plt.tight_layout();